# Entrenamiento en Colab (flujo híbrido)

Colab **entrena**; tu máquina local **evalúa y visualiza**. Checkpoints en Drive (persisten), logs en disco local de Colab y sincronizados a Drive cada pocos minutos.

Antes: *Entorno de ejecución → Cambiar tipo de entorno → GPU*. Sube `mario_rl_dia1.zip` a `MyDrive/mario/` (o usa un repo Git).

In [ ]:
# 1) GPU
!nvidia-smi
import torch; print("CUDA:", torch.cuda.is_available())

In [ ]:
# 2) Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3) Código del proyecto desde GitHub
REPO_URL = "https://github.com/David-AGL/mario-bros-reinforcement-learning.git"
BRANCH = "main"   # cambia si trabajas en otra rama

import os
if not os.path.exists("/content/mario/.git"):
    !git clone -b {BRANCH} {REPO_URL} /content/mario
else:
    !git -C /content/mario pull origin {BRANCH}
%cd /content/mario
!git log --oneline -1

In [ ]:
# 4) Dependencias (mismas versiones que requirements.txt, sin tocar torch de Colab)
!pip install -q "numpy<2" "gym==0.26.2" "nes-py==8.2.1" "gym-super-mario-bros==7.4.0" "gymnasium==0.29.1" "stable-baselines3==2.3.2" opencv-python-headless psutil tensorboard
# Si pip avisa de conflictos con NumPy o pide reiniciar: Entorno de ejecución -> Reiniciar sesión, y vuelve a ejecutar desde la celda 2 (omite esta).

In [ ]:
# 5) Smoke test del entorno + FPS en Colab (compáralo con tu portátil)
!python -W ignore scripts/smoke_test_env.py

## Configuración de la corrida
`TOTAL_STEPS` debe ser **el mismo para DQN y PPO**; fíjalo tras medir FPS.

In [ ]:
SEED = 1
TOTAL_STEPS = 50_000        # smoke test del día 2; luego el presupuesto real
RUN_NAME = f"dqn_s{SEED}"   # usa otro nombre para pruebas, p. ej. dqn_smoke

DRIVE = "/content/drive/MyDrive/mario"
MODELS_ROOT = f"{DRIVE}/models"          # checkpoints: persisten en Drive
LOGS_ROOT = "/content/runs"              # logs: disco local (rápido)
LOGS_DRIVE = f"{DRIVE}/runs"             # copia de seguridad de logs

# En Colab (~12 GB RAM gratis) conviene reducir el buffer. Ajusta tras ver el dry-run.
EXTRA = "--optimize-memory-usage"

In [ ]:
# 6) Dry-run: configuración + memoria estimada (no entrena)
!python -m dqn.train --seed {SEED} --total-steps {TOTAL_STEPS} --run-name {RUN_NAME} \
    --models-root {MODELS_ROOT} --logs-root {LOGS_ROOT} {EXTRA} --dry-run

In [ ]:
# 7) Sincronizar logs a Drive en segundo plano cada 5 min
import threading, time, subprocess, os
os.makedirs(LOGS_ROOT, exist_ok=True); os.makedirs(LOGS_DRIVE, exist_ok=True)
_stop = False
def _sync():
    while not _stop:
        subprocess.run(["rsync", "-a", f"{LOGS_ROOT}/", f"{LOGS_DRIVE}/"])
        time.sleep(300)
threading.Thread(target=_sync, daemon=True).start()

In [ ]:
# 8) ENTRENAR (si es una reanudación, usa la celda 9)
!python -W ignore -m dqn.train --seed {SEED} --total-steps {TOTAL_STEPS} --run-name {RUN_NAME} \
    --models-root {MODELS_ROOT} --logs-root {LOGS_ROOT} {EXTRA}

## Reanudar tras una caída
Busca el último checkpoint en Drive y reanuda con el **mismo** `RUN_NAME` y `TOTAL_STEPS`.
Sin replay buffer guardado, el buffer se vuelve a llenar (queda registrado en la salida).

In [ ]:
# 9) Reanudar
import glob, re
ckpts = glob.glob(f"{MODELS_ROOT}/dqn/{RUN_NAME}/dqn_s{SEED}_*_steps.zip")
last = max(ckpts, key=lambda p: int(re.search(r"_(\d+)_steps", p).group(1)))
print("Reanudando desde:", last)
!python -W ignore -m dqn.train --seed {SEED} --total-steps {TOTAL_STEPS} --run-name {RUN_NAME} \
    --models-root {MODELS_ROOT} --logs-root {LOGS_ROOT} {EXTRA} --resume {last}

In [ ]:
# 10) Al terminar: sincronizar logs y listar lo que hay en Drive
_stop = True
!rsync -a {LOGS_ROOT}/ {LOGS_DRIVE}/
!ls -lh {MODELS_ROOT}/dqn/{RUN_NAME}

## En tu máquina local
Descarga de Drive `mario/models/` y `mario/runs/` a las carpetas `models/` y `runs/` del repo. Luego:
`tensorboard --logdir runs/` y evalúa con `evaluate.py` (solo inferencia, funciona en CPU).